# Kigo — Google Colab launcher

Colab equivalent of `kaggle/kaggle_run.py`. Clones the repo, installs deps without disturbing Colab's CUDA-matched torch, pulls the tokenized dataset from an HF **dataset** repo, and trains — syncing checkpoints to the Hub so a disconnected session resumes by just rerunning the last cell.

**Before running:**
1. **Runtime → Change runtime type → GPU**.
2. Add these in the **Secrets** panel (left sidebar, key icon), each with *Notebook access* on:
   - `HF_TOKEN` — Hugging Face token (write).
   - `WANDB_API_KEY` — Weights & Biases key.
   - `REPO_URL` — e.g. `github.com/you/Kigo.git`.
   - `HF_CKPT_REPO` — checkpoint model repo, e.g. `you/kigo`.
   - `HF_DATA_REPO` — tokenized dataset repo, e.g. `you/kigo-fineweb-edu`.
   - `GITHUB_TOKEN` — *only* if the GitHub repo is private.
3. One-time: upload your tokenized `train/`, `val/`, `test/` splits to the dataset repo, e.g. `huggingface-cli upload you/kigo-fineweb-edu ./data --repo-type dataset`.

In [ ]:
# Confirm a GPU is attached (Runtime → Change runtime type → GPU).
!nvidia-smi

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import userdata

# --- Secrets (from the Colab Secrets panel) ---
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
repo_url = userdata.get("REPO_URL")        # e.g. github.com/you/Kigo.git
hf_repo = userdata.get("HF_CKPT_REPO")     # e.g. you/kigo
data_repo = userdata.get("HF_DATA_REPO")   # e.g. you/kigo-fineweb

# Private GitHub repo needs a GITHUB_TOKEN secret; a public repo clones without one.
try:
    repo_url = f"{userdata.get('GITHUB_TOKEN')}@{repo_url}"
except Exception:
    pass

# --- Clone + install (Colab ships a CUDA-matched torch; --no-deps keeps it) ---
if not Path("repo").exists():
    subprocess.run(["git", "clone", "--depth", "1", f"https://{repo_url}", "repo"], check=True)
subprocess.run(["pip", "install", "-e", ".", "--no-deps"], cwd="repo", check=True)
subprocess.run(["pip", "install", "lightning", "wandb", "huggingface_hub", "torchdata"], check=True)

# --- Pull the tokenized dataset from the HF dataset repo ---
from huggingface_hub import snapshot_download

root = Path(snapshot_download(repo_id=data_repo, repo_type="dataset", local_dir="/content/data"))
train = next((p for p in root.rglob("train") if p.is_dir()), None)
if train is None:
    raise FileNotFoundError(f"No train/ split under {root}")
data_dir = str(train.parent)

# --- Train (single GPU; resumes from the Hub if checkpoints exist) ---
subprocess.run(
    ["python", "train.py",
     "--config", "config/kigo-162m.yaml",
     "--data-dir", data_dir,
     "--hf-repo", hf_repo,
     "--auto-batch"],
    cwd="repo", check=True,
)